In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash-lite")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x000001A53E027470>, default_metadata=(), model_kwa

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x000001A53E027470>, default_metadata=(), model_kwa

In [5]:
model.invoke("Provide details about the movie Inception")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': '**Inception** is a 2010 science fiction action film written and directed by Christopher Nolan. Starring Leonardo DiCaprio, it is widely regarded as a modern classic known for its complex plot, mind-bending practical effects, and ambiguous ending. \n\nHere are the full details of the movie, broken down by plot, cast, themes, and production:\n\n---\n\n### **1. The Premise & Core Concept**\nThe film is set in a world where shared dreaming technology exists. Specifically, people can enter the dreams of others to extract information (espionage) or, in rarer cases, to plant an idea (inception). \n\nThe movie operates on several key rules regarding dreams within dreams:\n* **Levels:** You can dream inside a dream, creating multiple layers. Time moves progressively slower the deeper you go into the subconscious.\n* **Kicks:** A physical sensation (like falling or being jolted) wakes a dreamer up from a lower level.\n* **Totems:** Small physical obje

In [7]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

message output alnogside the parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("provide details about the movie inception")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', 'extras': {'signature': 'El4KXAERTTIP1OcIVfdKEk7dff9Vc5XB/WCH0YuPRSYcjp5Ig78xod4qZN9amY6DXUd/1iHpYNDIgDGZ7AuQGQi68pn89E2pMgqoVKxvuPt6COr/88QNkCl9XRgcRcnE'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a066ca-7c5a-75c1-9264-c731cf0be648-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 41, 'total_tokens': 48, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

nested structure 

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

###TYPEDICT

no runtime validation , a simple alternative using python built in typing       

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)

response = model_withtypedict.invoke(
    "Please provide details of the movie Avengers"
)

response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [17]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Sci-Fi', 'Superhero'],
 'budget': 220000000}

In [19]:
model.profile

{'name': 'Gemini 3.5 Flash Lite',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

data classes

In [20]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=init_chat_model("google_genai:gemini-3.5-flash-lite"),
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})
result
# print(result["structured_response"])

# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='1841dd08-a3f5-4776-849a-c216d801a5a0'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', 'extras': {'signature': 'El4KXAERTTIPIBE7oNlJjOC7JKnxazGQZCFBdZiXnE6sGGsrUwxxm/PqJgqy+kWbvXcSPUFNQTVy3M6RSQosME2lbfzIUyUHjki8TgzJCuFyD7wRhZIv72Wa4Qw+Xhgy'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a066dd-5c0f-7402-a585-0a408f6b0614-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [23]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [24]:
## TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

agent = create_agent(
    model=init_chat_model("google_genai:gemini-3.5-flash-lite"),
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

result["structured_response"]

# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

agent = create_agent(
    model=init_chat_model("google_genai:gemini-3.5-flash-lite"),
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')